#判断三款候选午餐袋的销量有多依赖单个大客户

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import URL
import os
from dotenv import load_dotenv

In [2]:
load_dotenv(r"D:\my_project\ecommerce-business-analytics\.env")
engine = create_engine(URL.create(
    "mysql+pymysql",
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", 3306)),
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    query={"charset": "utf8mb4"}
))

In [3]:
# 1.准备数据
data = pd.read_sql('select * from v_sales_transactions where invoice_date >= "2010-12-01" and invoice_date < "2011-12-01" and customer_id is not null and stock_code in ("20725", "22383", "20726")', engine)

In [4]:
# 2.计算出商品各自对应的总销量
total_quantity = data.groupby('stock_code')['quantity'].sum().reset_index(name='total_quantity')
total_quantity

,stock_code,total_quantity
0,20725,17540
1,20726,8945
2,22383,11604


In [5]:
# 3.计算购买量最大的单个客户购买量
t1 = data.groupby(['stock_code', 'customer_id'])['quantity'].sum().reset_index(name='single_max_quantity')
single_max = t1.groupby('stock_code')['single_max_quantity'].max()

In [6]:
# 4.合并表
finall_table = total_quantity.merge(single_max, on='stock_code', how='left')
# 5.计算占比保留两位小数
finall_table['max_ratio'] = ((finall_table['single_max_quantity'] / finall_table['total_quantity']) * 100).round(2)
finall_table

,stock_code,total_quantity,single_max_quantity,max_ratio
0,20725,17540,1300,7.41
1,20726,8945,1300,14.53
2,22383,11604,920,7.93


In [7]:
# 6.完成最终汇总
finall_table = finall_table.merge(data, how='inner', on='stock_code').drop_duplicates(subset='stock_code', keep='first')[['description','stock_code','total_quantity','single_max_quantity','max_ratio']]

In [10]:
# 7.前5大客户合计购买量
t2 = data.groupby(['stock_code','customer_id'])['quantity'].sum().reset_index(name='single_max_quantity')
# 写法1：
top5 = t2.sort_values('single_max_quantity', ascending=False).groupby('stock_code').head(5)
total_top5 = top5.groupby('stock_code')['single_max_quantity'].sum().reset_index(name='top5_quantity')
finall_table = total_top5.merge(finall_table, on='stock_code', how='inner')[['description','stock_code','total_quantity','single_max_quantity','max_ratio','top5_quantity']]

In [11]:
# 8.计算前5大客户合计购买量占比
finall_table['top5_ratio'] = ((finall_table['top5_quantity'] / finall_table['total_quantity']) * 100).round(2)
finall_table

,description,stock_code,total_quantity,single_max_quantity,max_ratio,top5_quantity,top5_ratio
0,LUNCH BAG RED RETROSPOT,20725,17540,1300,7.41,3429,19.55
1,LUNCH BAG WOODLAND,20726,8945,1300,14.53,2197,24.56
2,LUNCH BAG SUKI DESIGN,22383,11604,920,7.93,2190,18.87


建议优先对 RED RETROSPOT 做小规模复购推广测试。观察期内，RED 有 272 名可识别客户重复购买，比 SUKI DESIGN 的 222 名多 50 名。两款商品前五大客户的购买量占比分别为 19.55% 和 18.87%。考虑到本次试点更看重已有多少客户重复购买，我建议接受 RED 高出 0.68 个百分点的集中度，优先测试这款商品。